In [ ]:
import pandas as pd
import numpy as np
from loguru import logger

config_file_path: str = r"/home/user/data-da-ds-de/data_validation_pandas/data/configs/master_data_setup config.xlsx"
df_config: pd.DataFrame = pd.read_excel(config_file_path, sheet_name="InternalSuspendedList")
df_config = df_config.loc[df_config["column_name"] != "Note"]
df_config

In [ ]:
@logger.catch
def validate_column_names(check_column: str = "", origin_columns: list | pd.Series = []) -> bool:
    if check_column not in origin_columns:
        raise ValueError(f"Column '{check_column}' not found in '{origin_columns}'.")
        return False

    return True

In [ ]:
@logger.catch
def split_by_separators(df: pd.DataFrame, column_name: str, separators: str = "") -> pd.DataFrame:
    if not separators:
        logger.info(f"Column '{column_name}' does not have any separators.")
        return df

    if not isinstance(separators, str):
        raise ValueError(f"Separators must be a string, got '{type(separators)}'.")
        return

    if not validate_column_names(check_column=column_name, origin_columns=df.columns):
        return

    is_contain_separators = df[column_name].str.contains(separators)
    if not is_contain_separators.any():
        logger.info(f"Column '{column_name}' does not contain any separators '{separators}'. So return origin data.")
        return df

    df[column_name] = df[column_name].astype(str).str.split(separators)
    return df

In [ ]:
@logger.catch
def mapping_values(df: pd.DataFrame, column_name: str, mapping: dict) -> pd.DataFrame:
    if not validate_column_names(check_column=column_name, origin_columns=df.columns):
        return
    if not isinstance(mapping, dict):
        raise ValueError(f"Mapping must be a dictionary, got {type(mapping)}.")
        return
    if not mapping:
        logger.info(f"Column '{column_name}' does not have any mapping.")
        return df

    df[column_name] = df[column_name].map(mapping)
    return df

In [ ]:
# test
df = pd.DataFrame({
    "col1": [1, 2, 3],
    "col2": ["a", "b", "c"],
    "col3": ["d,e,f", "e", "f"]
    })

df = (
    df
    .pipe(split_by_separators, column_name="col2", separators=",")
    .pipe(split_by_separators, column_name="col3", separators=",")
    .pipe(mapping_values, column_name="col1", mapping={1: "one", 2: "two", 3: "three"})
)
df

In [ ]:
df_is_preprocessing = df_config.loc[df_config["is_preprocessing"] == True]

if not df_is_preprocessing.empty:
    for index, row in df_is_preprocessing.iterrows():
        df = df.pipe(split_by_separators, column_name=row["column_name"], separators=row["seperators"])
        df = df.pipe(mapping_values, column_name=row["column_name"], mapping=row["mapping"])

    logger.info("Data preprocessing completed.")

logger.info("No column needs preprocessing.")



In [ ]:
# required check
required_check_columns = df_config.loc[df_config["is_required_check"] == True, "column_name"]
logger.info(f"Required check columns ({len(required_check_columns)}): {required_check_columns.tolist()}, ")

if required_check_columns.empty:
    logger.info("No column needs required check.")

for column_name in required_check_columns:
    if column_name not in df.columns:
        logger.error(f"Column '{column_name}' is required but not found in the dataframe.")
    else:
        logger.info(f"Column '{column_name}' is required and found in the dataframe.")
        